In [1]:
%pip install -q pandas numpy requests plotly ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 45.1 MB/s eta 0:00:00


In [2]:

import os
import time
import json
import zipfile
import warnings
from pathlib import Path

import requests
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, HTML
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

warnings.filterwarnings("ignore", category=FutureWarning)

# Project settings
PROJECT_NAME = "Global_Economy_GDP_Analytics"
OUTPUT_DIR = Path("/content") / PROJECT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR = 2000
END_YEAR = 2025
DEFAULT_TOP_N = 15

# World Bank API
BASE_URL = "https://api.worldbank.org/v2"

# Consistent chart appearance
TEMPLATE = "plotly_dark"

print("Project folder:", OUTPUT_DIR)
print("Analysis period:", START_YEAR, "-", END_YEAR)
print("Libraries imported successfully!")


Project folder: /content/Global_Economy_GDP_Analytics
Analysis period: 2000 - 2025
Libraries imported successfully!


In [3]:

session = requests.Session()

retry_strategy = Retry(
    total=4,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

def world_bank_get(url, params=None, timeout=90):
    """Request JSON from the World Bank API."""
    response = session.get(
        url,
        params=params,
        timeout=timeout
    )
    response.raise_for_status()

    result = response.json()

    if not isinstance(result, list) or len(result) < 2:
        raise ValueError(
            f"Unexpected API response from {response.url}"
        )

    if result[1] is None:
        raise ValueError(
            f"No data returned from {response.url}"
        )

    return result

# Test the connection
test_response = world_bank_get(
    f"{BASE_URL}/country/ind/indicator/NY.GDP.MKTP.CD",
    params={
        "format": "json",
        "date": str(2023),
        "per_page": 5
    }
)

print("World Bank API connection successful!")
print("Response metadata:", test_response[0])


World Bank API connection successful!
Response metadata: {'page': 1, 'pages': 1, 'per_page': 5, 'total': 1, 'sourceid': '2', 'lastupdated': '2026-10-08'}


In [4]:

INDICATORS = {
    "gdp": "NY.GDP.MKTP.CD",
    "gdp_per_capita": "NY.GDP.PCAP.CD",
    "gdp_growth": "NY.GDP.MKTP.KD.ZG",
    "population": "SP.POP.TOTL"
}

def download_indicator(indicator_code, start_year, end_year):
    """Download all available pages for one indicator."""

    url = f"{BASE_URL}/country/all/indicator/{indicator_code}"

    params = {
        "format": "json",
        "date": f"{start_year}:{end_year}",
        "per_page": 20000,
        "page": 1
    }

    first = world_bank_get(url, params=params)
    metadata = first[0]
    records = list(first[1] or [])

    total_pages = int(metadata.get("pages", 1))

    for page in range(2, total_pages + 1):
        params["page"] = page
        response = world_bank_get(url, params=params)
        records.extend(response[1] or [])

    if not records:
        raise ValueError(
            f"No observations received for {indicator_code}"
        )

    result = pd.DataFrame(records)

    result["year"] = pd.to_numeric(
        result["date"], errors="coerce"
    )
    result["value"] = pd.to_numeric(
        result["value"], errors="coerce"
    )

    result = result.rename(columns={
        "countryiso3code": "country_code",
        "country": "country_info"
    })

    result["country"] = result["country_info"].apply(
        lambda item: item.get("value")
        if isinstance(item, dict) else None
    )

    result = result[
        ["country_code", "country", "year", "value"]
    ].dropna(subset=["country_code", "year"])

    result["year"] = result["year"].astype(int)

    return result


indicator_frames = {}

for name, code in INDICATORS.items():
    print(f"Downloading {name} ({code})...")

    indicator_frames[name] = download_indicator(
        code, START_YEAR, END_YEAR
    )

print("\nAll four indicators downloaded.")
for name, frame in indicator_frames.items():
    print(f"{name}: {len(frame):,} observations")



All four indicators downloaded.
gdp: 6,916 observations
gdp_per_capita: 6,916 observations
gdp_growth: 6,916 observations
population: 6,916 observations


In [5]:

country_response = world_bank_get(
    f"{BASE_URL}/country",
    params={
        "format": "json",
        "per_page": 400,
        "page": 1
    }
)

country_records = list(country_response[1] or [])

total_pages = int(country_response[0].get("pages", 1))

for page in range(2, total_pages + 1):
    response = world_bank_get(
        f"{BASE_URL}/country",
        params={
            "format": "json",
            "per_page": 400,
            "page": page
        }
    )
    country_records.extend(response[1] or [])

metadata_rows = []

for item in country_records:
    region = item.get("region") or {}
    income = item.get("incomeLevel") or {}

    metadata_rows.append({
        "country_code": item.get("id"),
        "country_meta": item.get("name"),
        "region": region.get("value"),
        "region_code": region.get("id"),
        "income_group": income.get("value"),
        "capital": item.get("capitalCity"),
        "longitude": item.get("longitude"),
        "latitude": item.get("latitude")
    })

country_metadata = pd.DataFrame(metadata_rows)

# Remove regional/income aggregates from country-level analysis.
# The World Bank assigns actual countries a non-empty region code.
country_metadata = country_metadata[
    country_metadata["region_code"].notna()
    & country_metadata["region_code"].astype(str).str.strip().ne("")
    & country_metadata["country_code"].notna()
].copy()

country_metadata = country_metadata.drop_duplicates(
    subset=["country_code"]
)

print("Country metadata loaded:", len(country_metadata))
display(country_metadata.head(10))


Country metadata loaded: 296


,country_code,country_meta,region,region_code,income_group,capital,longitude,latitude
0,ABW,Aruba,Latin America & Caribbean,LCN,High income,Oranjestad,-70.0167,12.5167
1,AFE,Africa Eastern and Southern,Aggregates,NA,Aggregates,,,
2,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228
3,AFR,Africa,Aggregates,NA,Aggregates,,,
4,AFW,Africa Western and Central,Aggregates,NA,Aggregates,,,
5,AGO,Angola,Sub-Saharan Africa,SSF,Lower middle income,Luanda,13.242,-8.81155
6,ALB,Albania,Europe & Central Asia,ECS,Upper middle income,Tirane,19.8172,41.3317
7,AND,Andorra,Europe & Central Asia,ECS,High income,Andorra la Vella,1.5218,42.5075
8,ARB,Arab World,Aggregates,NA,Aggregates,,,
9,ARE,United Arab Emirates,"Middle East, North Africa, Afghanistan & Pakistan",MEA,High income,Abu Dhabi,54.3705,24.4764


In [6]:

# Start with the GDP dataset
gdp_df = indicator_frames["gdp"].rename(
    columns={"value": "gdp"}
)

data = gdp_df[["country_code", "country", "year", "gdp"]].copy()

# Merge the remaining indicators
for name in ["gdp_per_capita", "gdp_growth", "population"]:
    temp = indicator_frames[name][
        ["country_code", "year", "value"]
    ].rename(columns={"value": name})

    data = data.merge(
        temp,
        on=["country_code", "year"],
        how="outer"
    )

# Attach country metadata
data = data.merge(
    country_metadata.drop(
        columns=["country_meta"], errors="ignore"
    ),
    on="country_code",
    how="left"
)

# Normalize country names where metadata is available
data["country"] = data["country"].fillna(
    data.get("country_meta", pd.Series(index=data.index, dtype="object"))
)

# Remove aggregate rows, if any slipped through
data = data[
    data["region_code"].notna()
    & data["region_code"].astype(str).str.strip().ne("")
].copy()

# Restrict analysis period and valid observations
data["year"] = pd.to_numeric(data["year"], errors="coerce")
data = data[
    data["year"].between(START_YEAR, END_YEAR)
    & data["country_code"].notna()
    & data["country"].notna()
].copy()

data["year"] = data["year"].astype(int)

numeric_columns = [
    "gdp", "gdp_per_capita", "gdp_growth", "population"
]

for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors="coerce")
    data.loc[~np.isfinite(data[col]), col] = np.nan

data = data.drop_duplicates(
    subset=["country_code", "year"]
).sort_values(["country", "year"])

# Derived column for readable charts
data["gdp_trillion_usd"] = data["gdp"] / 1e12

data.to_csv(
    OUTPUT_DIR / "global_economy_all_years.csv",
    index=False
)

print("Cleaned dataset shape:", data.shape)
print("Countries:", data["country_code"].nunique())
print("Years:", data["year"].min(), "to", data["year"].max())
print("\nMissing values:")
display(data[numeric_columns].isna().sum().to_frame("missing_count"))

display(data.head(10))


Cleaned dataset shape: (6786, 14)
Countries: 261
Years: 2000 to 2025

Missing values:


,missing_count
gdp,214
gdp_per_capita,214
gdp_growth,288
population,0


,country_code,country,year,gdp,gdp_per_capita,gdp_growth,population,region,region_code,income_group,capital,longitude,latitude,gdp_trillion_usd
16302,AFG,Afghanistan,2000,3.521418e+09,174.930991,NaN,20130327.0,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228,0.003521
16303,AFG,Afghanistan,2001,2.813572e+09,138.706822,-9.431974,20284307.0,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228,0.002814
16304,AFG,Afghanistan,2002,3.825701e+09,178.954088,28.600001,21378117.0,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228,0.003826
16305,AFG,Afghanistan,2003,4.520947e+09,198.871116,8.832278,22733049.0,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228,0.004521
16306,AFG,Afghanistan,2004,5.224897e+09,221.763654,1.414118,23560654.0,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228,0.005225
16307,AFG,Afghanistan,2005,6.203257e+09,254.184249,11.229715,24404567.0,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228,0.006203
16308,AFG,Afghanistan,2006,6.971758e+09,274.218554,5.357403,25424094.0,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228,0.006972
16309,AFG,Afghanistan,2007,9.747886e+09,376.223152,13.826320,25909852.0,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228,0.009748
16310,AFG,Afghanistan,2008,1.010930e+10,381.733238,3.924984,26482622.0,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228,0.010109
16311,AFG,Afghanistan,2009,1.241615e+10,452.053705,21.390528,27466101.0,"Middle East, North Africa, Afghanistan & Pakistan",MEA,Low income,Kabul,69.1761,34.5228,0.012416


In [7]:
data["country"] = data["country"].fillna("")

In [9]:

coverage = (
    data.dropna(subset=["gdp"])
    .groupby("year")["country_code"]
    .nunique()
)

if coverage.empty:
    raise ValueError("No valid GDP observations were found.")

best_coverage = coverage.max()

eligible_years = coverage[
    coverage >= 0.85 * best_coverage
].index

LATEST_YEAR = int(max(eligible_years))

latest_df = data[
    (data["year"] == LATEST_YEAR)
    & data["gdp"].notna()
].copy()

latest_df = latest_df.sort_values("gdp", ascending=False)

# Summary indicators
total_gdp = latest_df["gdp"].sum(min_count=1)
country_count = latest_df["country_code"].nunique()

top_country = latest_df.iloc[0]["country"]
top_country_gdp = latest_df.iloc[0]["gdp"]

median_gdp_pc = latest_df["gdp_per_capita"].median()

display(HTML(f"""
<div style="
    display:grid;
    grid-template-columns:repeat(auto-fit,minmax(180px,1fr));
    gap:12px;
    font-family:Arial;
">
  <div style="background:#111827;color:white;padding:18px;border-radius:12px">
    <div style="color:#9CA3AF">Latest well-covered year</div>
    <h2>{LATEST_YEAR}</h2>
  </div>
  <div style="background:#111827;color:white;padding:18px;border-radius:12px">
    <div style="color:#9CA3AF">Combined country GDP</div>
    <h2>${total_gdp / 1e12:,.2f}T</h2>
  </div>
  <div style="background:#111827;color:white;padding:18px;border-radius:12px">
    <div style="color:#9CA3AF">Countries/economies</div>
    <h2>{country_count:,}</h2>
  </div>
  <div style="background:#111827;color:white;padding:18px;border-radius:12px">
    <div style="color:#9CA3AF">Largest economy in dataset</div>
    <h3>{top_country}</h3>
    <div>${top_country_gdp / 1e12:,.2f}T</div>
  </div>
</div>
"""))

print("Median GDP per capita: ",
      f"${median_gdp_pc:,.0f}" if pd.notna(median_gdp_pc) else "Unavailable")
print("GDP coverage:", int(coverage.loc[LATEST_YEAR]), "countries")


Median GDP per capita:  $7,822
GDP coverage: 230 countries


In [10]:

available_years = sorted(
    data.loc[data["gdp"].notna(), "year"].unique().tolist()
)

map_metric = widgets.Dropdown(
    options=[
        ("Total GDP (trillion current US$)", "gdp_trillion_usd"),
        ("GDP per capita (current US$)", "gdp_per_capita"),
        ("Annual GDP growth (%)", "gdp_growth")
    ],
    value="gdp_trillion_usd",
    description="Metric:",
    layout=widgets.Layout(width="420px")
)

map_year = widgets.SelectionSlider(
    options=available_years,
    value=LATEST_YEAR,
    description="Year:",
    continuous_update=False,
    layout=widgets.Layout(width="650px")
)

def show_world_map(year, metric):
    frame = data[data["year"] == year].copy()
    frame = frame.dropna(subset=[metric])

    labels = {
        "gdp_trillion_usd": "GDP (trillion US$)",
        "gdp_per_capita": "GDP per capita (US$)",
        "gdp_growth": "GDP growth (%)"
    }

    fig = px.choropleth(
        frame,
        locations="country_code",
        color=metric,
        hover_name="country",
        hover_data={
            "country_code": True,
            "region": True,
            "gdp": ":,.0f",
            "gdp_per_capita": ":,.0f",
            "gdp_growth": ":.2f",
            "population": ":,.0f",
            metric: ":.3f"
        },
        locationmode="ISO-3",
        color_continuous_scale="Viridis",
        title=f"Global Economy Map — {labels[metric]} ({year})"
    )

    fig.update_layout(
        template=TEMPLATE,
        height=650,
        geo=dict(showframe=False, showcoastlines=True)
    )

    fig.show()

display(map_metric, map_year)
map_output = widgets.interactive_output(
    show_world_map,
    {"year": map_year, "metric": map_metric}
)
display(map_output)


Dropdown(description='Metric:', layout=Layout(width='420px'), options=(('Total GDP (trillion current US$)', 'g…

SelectionSlider(continuous_update=False, description='Year:', index=25, layout=Layout(width='650px'), options=…

Output()

In [11]:

region_options = ["All regions"] + sorted(
    latest_df["region"].dropna().unique().tolist()
)

region_selector = widgets.Dropdown(
    options=region_options,
    value="All regions",
    description="Region:"
)

top_n_selector = widgets.IntSlider(
    value=15,
    min=5,
    max=30,
    step=5,
    description="Top N:",
    continuous_update=False
)

def show_gdp_ranking(region, top_n):
    frame = data[data["year"] == LATEST_YEAR].copy()
    frame = frame.dropna(subset=["gdp"])

    if region != "All regions":
        frame = frame[frame["region"] == region]

    frame = frame.nlargest(top_n, "gdp").sort_values("gdp")

    if frame.empty:
        print("No data available for this selection.")
        return

    frame["GDP (trillion US$)"] = frame["gdp"] / 1e12

    fig = px.bar(
        frame,
        x="GDP (trillion US$)",
        y="country",
        orientation="h",
        color="GDP (trillion US$)",
        color_continuous_scale="Teal",
        hover_data=["gdp_per_capita", "gdp_growth", "population"],
        title=f"Top {len(frame)} Economies — {LATEST_YEAR}"
    )

    fig.update_layout(
        template=TEMPLATE,
        height=max(450, len(frame) * 28),
        yaxis_title="Country",
        coloraxis_showscale=False
    )

    fig.show()

display(region_selector, top_n_selector)

ranking_output = widgets.interactive_output(
    show_gdp_ranking,
    {"region": region_selector, "top_n": top_n_selector}
)
display(ranking_output)


Dropdown(description='Region:', options=('All regions', 'Aggregates', 'East Asia & Pacific', 'Europe & Central…

IntSlider(value=15, continuous_update=False, description='Top N:', max=30, min=5, step=5)

Output()

In [12]:

country_names = sorted(data["country"].dropna().unique().tolist())

default_countries = [
    country for country in [
        "United States", "China", "India",
        "Japan", "Germany", "United Kingdom"
    ]
    if country in country_names
]

country_selector = widgets.SelectMultiple(
    options=country_names,
    value=tuple(default_countries),
    description="Countries:",
    rows=8,
    layout=widgets.Layout(width="400px")
)

def show_gdp_trends(countries):
    if not countries:
        print("Select at least one country.")
        return

    frame = data[
        data["country"].isin(list(countries))
        & data["gdp"].notna()
    ].copy()

    frame["GDP (trillion US$)"] = frame["gdp"] / 1e12

    fig = px.line(
        frame,
        x="year",
        y="GDP (trillion US$)",
        color="country",
        markers=True,
        title="Historical GDP Trends",
        hover_data=["gdp_per_capita", "population"]
    )

    fig.update_layout(
        template=TEMPLATE,
        height=560,
        xaxis_title="Year",
        yaxis_title="GDP (trillion current US$)",
        legend_title="Country"
    )

    fig.show()

display(country_selector)

trend_output = widgets.interactive_output(
    show_gdp_trends,
    {"countries": country_selector}
)
display(trend_output)


SelectMultiple(description='Countries:', index=(249, 46, 110, 119, 88, 248), layout=Layout(width='400px'), opt…

Output()

In [13]:

treemap_df = latest_df.dropna(
    subset=["region", "gdp"]
).copy()

treemap_df = treemap_df[treemap_df["gdp"] > 0]

fig_treemap = px.treemap(
    treemap_df,
    path=["region", "country"],
    values="gdp",
    color="gdp_per_capita",
    color_continuous_scale="Plasma",
    hover_data={
        "gdp": ":,.0f",
        "gdp_per_capita": ":,.0f",
        "gdp_growth": ":.2f",
        "population": ":,.0f"
    },
    title=f"Global GDP Treemap — {LATEST_YEAR}"
)

fig_treemap.update_layout(
    template=TEMPLATE,
    height=700
)

fig_treemap.show()


In [14]:

growth_df = latest_df.dropna(subset=["gdp_growth"]).copy()

if growth_df.empty:
    print("GDP growth data is unavailable for this year.")
else:
    fastest = growth_df.nlargest(10, "gdp_growth")
    slowest = growth_df.nsmallest(10, "gdp_growth")

    combined_growth = pd.concat(
        [fastest, slowest]
    ).drop_duplicates("country_code")

    combined_growth = combined_growth.sort_values("gdp_growth")

    fig_growth = px.bar(
        combined_growth,
        x="gdp_growth",
        y="country",
        orientation="h",
        color="gdp_growth",
        color_continuous_scale="RdYlGn",
        color_continuous_midpoint=0,
        hover_data=["gdp", "gdp_per_capita"],
        title=f"Highest and Lowest Recorded GDP Growth — {LATEST_YEAR}"
    )

    fig_growth.add_vline(
        x=0,
        line_dash="dash",
        line_color="white"
    )

    fig_growth.update_layout(
        template=TEMPLATE,
        height=650,
        xaxis_title="Annual GDP growth (%)",
        yaxis_title="Country",
        coloraxis_showscale=False
    )

    fig_growth.show()


In [15]:

insights = []

ranked = latest_df.dropna(subset=["gdp"]).sort_values(
    "gdp", ascending=False
)

if not ranked.empty:
    largest = ranked.iloc[0]
    insights.append(
        f"Largest economy in the dataset for {LATEST_YEAR}: "
        f"{largest['country']} "
        f"(GDP ${largest['gdp']/1e12:.2f} trillion)."
    )

    if len(ranked) >= 5:
        top_five_share = ranked.head(5)["gdp"].sum() / ranked["gdp"].sum() * 100
        insights.append(
            f"The top five economies account for approximately "
            f"{top_five_share:.1f}% of GDP in the included country sample."
        )

per_capita_df = latest_df.dropna(
    subset=["gdp_per_capita"]
).sort_values("gdp_per_capita", ascending=False)

if not per_capita_df.empty:
    richest_pc = per_capita_df.iloc[0]
    insights.append(
        f"Highest recorded GDP per capita in the dataset: "
        f"{richest_pc['country']} "
        f"(${richest_pc['gdp_per_capita']:,.0f})."
    )

growth_available = latest_df.dropna(subset=["gdp_growth"])

if not growth_available.empty:
    fastest = growth_available.loc[
        growth_available["gdp_growth"].idxmax()
    ]
    slowest = growth_available.loc[
        growth_available["gdp_growth"].idxmin()
    ]

    insights.append(
        f"Highest recorded annual GDP growth: {fastest['country']} "
        f"({fastest['gdp_growth']:.2f}%)."
    )
    insights.append(
        f"Lowest recorded annual GDP growth: {slowest['country']} "
        f"({slowest['gdp_growth']:.2f}%)."
    )

regional = latest_df.dropna(subset=["region", "gdp"]).groupby(
    "region", as_index=False
)["gdp"].sum()

if not regional.empty:
    largest_region = regional.loc[regional["gdp"].idxmax()]
    insights.append(
        f"Largest regional subtotal in the included data: "
        f"{largest_region['region']} "
        f"(${largest_region['gdp']/1e12:.2f} trillion)."
    )

display(HTML(
    "<h2>📊 Automatically Generated Insights</h2>"
    + "".join(
        f"<div style='padding:12px;margin:8px 0;"
        f"border-left:4px solid #14b8a6;"
        f"background:#111827;color:white;border-radius:5px'>"
        f"{i}. {insight}</div>"
        for i, insight in enumerate(insights, start=1)
    )
))

print("\nInterpretation note:")
print(
    "These are descriptive comparisons. GDP per capita is an average, "
    "not an individual salary, and current-US-dollar GDP is not "
    "adjusted for inflation or purchasing-power differences."
)



Interpretation note:
These are descriptive comparisons. GDP per capita is an average, not an individual salary, and current-US-dollar GDP is not adjusted for inflation or purchasing-power differences.


In [16]:

top_20_gdp = latest_df.nlargest(20, "gdp")[
    [
        "country", "region", "income_group",
        "gdp", "gdp_per_capita", "gdp_growth", "population"
    ]
].copy()

top_20_gdp["gdp_trillion_usd"] = top_20_gdp["gdp"] / 1e12

top_20_gdp = top_20_gdp.drop(columns=["gdp"])

top_20_per_capita = latest_df.dropna(
    subset=["gdp_per_capita"]
).nlargest(20, "gdp_per_capita")[
    [
        "country", "region", "gdp_per_capita",
        "gdp", "gdp_growth", "population"
    ]
]

regional_summary = latest_df.groupby(
    "region", as_index=False
).agg(
    gdp_total=("gdp", "sum"),
    median_gdp_per_capita=("gdp_per_capita", "median"),
    median_growth=("gdp_growth", "median"),
    country_count=("country_code", "nunique")
).sort_values("gdp_total", ascending=False)

print("TOP 20 ECONOMIES")
display(top_20_gdp)

print("TOP 20 BY GDP PER CAPITA")
display(top_20_per_capita)

print("REGIONAL SUMMARY")
display(regional_summary)


TOP 20 ECONOMIES


,country,region,income_group,gdp_per_capita,gdp_growth,population,gdp_trillion_usd
22879,World,Aggregates,Aggregates,14405.848460,2.921273,8.215425e+09,118.350166
20877,OECD members,Aggregates,Aggregates,51733.714020,1.770059,1.402244e+09,72.543268
21319,Post-demographic dividend,Aggregates,Aggregates,59490.024063,1.653310,1.132622e+09,67.379697
18927,IDA & IBRD total,Aggregates,Aggregates,6613.209470,4.335706,6.993791e+09,46.251403
18901,IBRD only,Aggregates,Aggregates,8599.358349,4.305323,5.006927e+09,43.056359
19811,Low & middle income,Aggregates,Aggregates,6107.261694,4.532801,6.791685e+09,41.478598
20227,Middle income,Aggregates,Aggregates,6778.797733,4.521788,6.024195e+09,40.836798
17913,East Asia & Pacific,Aggregates,Aggregates,14077.271203,3.997341,2.390874e+09,33.656981
20591,North America,Aggregates,Aggregates,86308.071727,2.129724,3.835011e+08,33.099237
17965,Europe & Central Asia,Aggregates,Aggregates,34603.480892,1.703297,9.325250e+08,32.268612


TOP 20 BY GDP PER CAPITA


,country,region,gdp_per_capita,gdp,gdp_growth,population
19915,Luxembourg,Europe & Central Asia,147252.179122,1.011578e+11,0.637878,6.869700e+05
19109,Ireland,Europe & Central Asia,131592.462548,7.217014e+11,12.337599,5.484367e+06
17237,Switzerland,Europe & Central Asia,114769.012314,1.043530e+12,1.300078,9.092436e+06
21579,Singapore,East Asia & Pacific,98813.978817,6.038695e+11,5.025882,6.111175e+06
19187,Iceland,Europe & Central Asia,98323.484952,3.858253e+10,1.279462,3.924040e+05
20773,Norway,Europe & Central Asia,94594.192957,5.307557e+11,1.085282,5.610870e+06
22671,United States,North America,90026.516301,3.076970e+13,2.161382,3.417849e+08
20591,North America,Aggregates,86308.071727,3.309924e+13,2.129724,3.835011e+08
17783,Denmark,Europe & Central Asia,76970.153522,4.625267e+11,2.928433,6.009169e+06
19967,"Macao SAR, China",East Asia & Pacific,75901.746129,5.206101e+10,4.671067,6.859000e+05


REGIONAL SUMMARY


,region,gdp_total,median_gdp_per_capita,median_growth,country_count
0,Aggregates,7.412741e+14,7161.606045,4.067495,44
5,North America,3.308960e+13,72862.090133,1.952034,2
1,East Asia & Pacific,3.268707e+13,6642.004495,3.617151,31
2,Europe & Central Asia,3.221258e+13,27761.744675,2.100844,50
3,Latin America & Caribbean,7.181822e+12,11059.208153,2.748317,33
6,South Asia,4.578022e+12,3597.631937,5.632179,6
4,"Middle East, North Africa, Afghanistan & Pakistan",4.494852e+12,5730.341112,3.589901,18
7,Sub-Saharan Africa,2.201297e+12,1351.359482,4.554075,46


In [17]:

# Save tables
top_20_gdp.to_csv(
    OUTPUT_DIR / "top_20_economies.csv", index=False
)

top_20_per_capita.to_csv(
    OUTPUT_DIR / "top_20_gdp_per_capita.csv", index=False
)

regional_summary.to_csv(
    OUTPUT_DIR / "regional_summary.csv", index=False
)

latest_df.to_csv(
    OUTPUT_DIR / "latest_year_country_data.csv", index=False
)

# Save insights
with open(
    OUTPUT_DIR / "economic_insights.txt",
    "w",
    encoding="utf-8"
) as file:
    file.write(f"GLOBAL ECONOMY & GDP ANALYTICS\n")
    file.write(f"Analysis year: {LATEST_YEAR}\n\n")
    file.write("\n".join(
        f"{i}. {insight}"
        for i, insight in enumerate(insights, start=1)
    ))
    file.write(
        "\n\nSource: World Bank World Development Indicators API.\n"
        "GDP is in current US dollars unless stated otherwise.\n"
    )

# Build a list of interactive figures for standalone HTML export.
# Use figures generated from the current dataset, not notebook widget outputs.
export_figures = {}

export_figures["gdp_map"] = px.choropleth(
    latest_df.dropna(subset=["gdp"]),
    locations="country_code",
    locationmode="ISO-3",
    color="gdp",
    hover_name="country",
    hover_data=["gdp_per_capita", "gdp_growth", "population"],
    color_continuous_scale="Viridis",
    title=f"World GDP Map — {LATEST_YEAR}"
).update_layout(template=TEMPLATE)

export_figures["gdp_ranking"] = px.bar(
    latest_df.nlargest(20, "gdp").sort_values("gdp"),
    x="gdp",
    y="country",
    orientation="h",
    title=f"Top 20 Economies — {LATEST_YEAR}"
).update_layout(template=TEMPLATE)

export_figures["gdp_per_capita_scatter"] = fig_scatter
export_figures["gdp_treemap"] = fig_treemap

if "fig_growth" in globals():
    export_figures["gdp_growth"] = fig_growth

# Write individual HTML charts
for name, fig in export_figures.items():
    fig.write_html(
        OUTPUT_DIR / f"{name}.html",
        include_plotlyjs=True,
        full_html=True
    )

# Create an HTML report that links the figures in one page.
html_parts = [
    "<!DOCTYPE html><html><head><meta charset='utf-8'>",
    "<title>Global Economy & GDP Analytics</title>",
    "<style>body{font-family:Arial;max-width:1200px;"
    "margin:30px auto;padding:0 16px;background:#0b1120;"
    "color:#f3f4f6}a{color:#5eead4}li{margin:12px 0}</style>",
    "</head><body>",
    "<h1>🌍 Global Economy & GDP Analytics</h1>",
    f"<p>Analysis year: {LATEST_YEAR}</p>",
    "<h2>Interactive visualizations</h2><ul>"
]

for name in export_figures:
    html_parts.append(
        f"<li><a href='{name}.html'>{name.replace('_', ' ').title()}</a></li>"
    )

html_parts.extend([
    "</ul><h2>Key insights</h2><ul>",
    *[f"<li>{insight}</li>" for insight in insights],
    "</ul>",
    "<p>Source: World Bank World Development Indicators API.</p>",
    "</body></html>"
])

(OUTPUT_DIR / "index.html").write_text(
    "\n".join(html_parts),
    encoding="utf-8"
)

# Bundle all project outputs
zip_path = Path("/content/Global_Economy_GDP_Analytics.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for file_path in OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            archive.write(
                file_path,
                arcname=str(file_path.relative_to(OUTPUT_DIR))
            )

print("Export completed!")
print("Project folder:", OUTPUT_DIR)
print("ZIP archive:", zip_path)
print("\nFiles created:")
for file_path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", file_path.name)


NameError: name 'fig_scatter' is not defined

In [18]:

from google.colab import files

files.download("/content/Global_Economy_GDP_Analytics.zip")


FileNotFoundError: Cannot find file: /content/Global_Economy_GDP_Analytics.zip